# Downloads

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e7_path = kagglehub.competition_download('playground-series-s6e7')

print('Data source import complete.')


100%|██████████| 22.3M/22.3M [00:00<00:00, 76.2MB/s]

Extracting files...


Data source import complete.


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e7')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/playground-series-s6e7


# Imports

In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [5]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.2 MB/s eta 0:00:00


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from catboost import CatBoostClassifier
import optuna
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier, early_stopping

# Data

In [17]:
train_df = pd.read_csv('/root/.cache/kagglehub/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/root/.cache/kagglehub/competitions/playground-series-s6e7/test.csv')
sample_submission = pd.read_csv('/root/.cache/kagglehub/competitions/playground-series-s6e7/sample_submission.csv')
sample_submission.head()

,id,health_condition
0,690088,at-risk
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,at-risk


In [18]:
train_df.health_condition.value_counts()

,count
health_condition,
at-risk,592561
unhealthy,57724
fit,39803


In [19]:
X_train = train_df.drop(columns=["health_condition", "id"])
X_test = test_df.drop(columns=["id"])
y_train = train_df["health_condition"]

# Preprocessing

In [20]:
categorical = [
    "diet_type",
    "stress_level",
    "sleep_quality",
    "physical_activity_level",
    "smoking_alcohol",
    "gender"
]


numerical = [
    "sleep_duration",
    "heart_rate",
    "bmi",
    "calorie_expenditure",
    "step_count",
    "exercise_duration",
    "water_intake"
]

X_train[categorical] = X_train[categorical].fillna("Missing").astype(str)
X_train[numerical] = X_train[numerical].apply(pd.to_numeric, errors="coerce")

X_test[categorical] = X_test[categorical].fillna("Missing").astype(str)
X_test[numerical] = X_test[numerical].apply(pd.to_numeric, errors="coerce")

# CatBoost

## Baseline CatBoost

In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    task_type="GPU",
    devices="0",
    verbose=10
)

In [14]:
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):

    print(f"\nFold {fold+1}")

    X_train_ = X_train.iloc[train_idx]
    X_val_ = X_train.iloc[val_idx]

    y_train_ = y_train.iloc[train_idx]
    y_val_ = y_train.iloc[val_idx]


    model.fit(
        X_train_,
        y_train_,
        cat_features=categorical
    )

    preds = model.predict(X_val_)

    score = f1_score(
        y_val_,
        preds,
        average="macro"
    )

    fold_scores.append(score)

    print("Macro F1:", score)


Fold 1
0:	learn: 0.9984424	total: 30.4ms	remaining: 3.01s
10:	learn: 0.4848848	total: 276ms	remaining: 2.23s
20:	learn: 0.2908169	total: 447ms	remaining: 1.68s
30:	learn: 0.2246076	total: 599ms	remaining: 1.33s
40:	learn: 0.2032772	total: 745ms	remaining: 1.07s
50:	learn: 0.1943277	total: 890ms	remaining: 856ms
60:	learn: 0.1886641	total: 1.03s	remaining: 661ms
70:	learn: 0.1845475	total: 1.18s	remaining: 483ms
80:	learn: 0.1816797	total: 1.36s	remaining: 320ms
90:	learn: 0.1799530	total: 1.51s	remaining: 150ms
99:	learn: 0.1785385	total: 1.64s	remaining: 0us
Macro F1: 0.858930218054632

Fold 2
0:	learn: 0.9985851	total: 23.6ms	remaining: 2.34s
10:	learn: 0.4712768	total: 349ms	remaining: 2.82s
20:	learn: 0.2885872	total: 792ms	remaining: 2.98s
30:	learn: 0.2241091	total: 1.33s	remaining: 2.97s
40:	learn: 0.2027029	total: 1.87s	remaining: 2.7s
50:	learn: 0.1930177	total: 2.36s	remaining: 2.27s
60:	learn: 0.1879985	total: 2.84s	remaining: 1.82s
70:	learn: 0.1840698	total: 3.29s	remaini

In [15]:
print("\nFold scores:", fold_scores)
print("Mean Macro F1:", np.mean(fold_scores))
print("Std Macro F1:", np.std(fold_scores))


Fold scores: [0.858930218054632, 0.8588042631654186, 0.8566755526570281, 0.8583420266611247, 0.8590623301068475]
Mean Macro F1: 0.8583628781290102
Std Macro F1: 0.0008778872869546379


## Hyper parameter tuned CatBoost for f1 score

> Add blockquote



In [16]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "MultiClass",
        "auto_class_weights": "Balanced",
        "task_type": "GPU",
        "devices": "0",
        "random_seed": 42,
        "verbose": False
    }

    scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=2000,
            **params
        )

        model.fit(
            X_tr, y_tr,
            cat_features=categorical,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            use_best_model=True
        )

        preds = model.predict(X_val)
        score = f1_score(y_val, preds, average="macro")
        scores.append(score)

        trial.report(np.mean(scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

In [17]:
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best macro F1:", study.best_value)
print("Best params:", study.best_params)

[I 2026-07-24 12:44:16,168] A new study created in memory with name: no-name-090c26c2-895e-4a73-b20a-058c1046c492


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-24 12:45:49,588] Trial 0 finished with value: 0.8640409945448638 and parameters: {'learning_rate': 0.08422527679586371, 'depth': 10, 'l2_leaf_reg': 9.575889295279383, 'border_count': 152, 'random_strength': 0.6673055579821663, 'bagging_temperature': 0.7681049275626523}. Best is trial 0 with value: 0.8640409945448638.
[I 2026-07-24 12:46:50,794] Trial 1 finished with value: 0.8611565109014009 and parameters: {'learning_rate': 0.1343390164411347, 'depth': 9, 'l2_leaf_reg': 5.137805276301005, 'border_count': 40, 'random_strength': 0.8274098728058965, 'bagging_temperature': 0.29102018691314313}. Best is trial 0 with value: 0.8640409945448638.
[I 2026-07-24 12:47:58,402] Trial 2 finished with value: 0.8639319318409788 and parameters: {'learning_rate': 0.12471207892731896, 'depth': 7, 'l2_leaf_reg': 9.000462130400528, 'border_count': 130, 'random_strength': 4.530962451138444, 'bagging_temperature': 0.5760507799724114}. Best is trial 0 with value: 0.8640409945448638.
[I 2026-07-24 

In [24]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score


# F1 optimised parameters
best_params = {
    "learning_rate": 0.03289045838210247,
    "depth": 9,
    "l2_leaf_reg": 3.540177939019512,
    "border_count": 209,
    "random_strength": 7.089965381625723,
    "bagging_temperature": 0.20482493896236287
}


final_params = {
    **best_params,

    "loss_function": "MultiClass",

    # keep this consistent with what you tuned
    "eval_metric": "TotalF1",

    "task_type": "GPU",
    "devices": "0",

    "verbose": False
}


seeds = [
    42,
    123,
    456,
    789,
    999
]


models = []
test_probs = []


for seed in seeds:

    print(f"Training seed {seed}")

    params = {
        **final_params,
        "random_seed": seed
    }


    model = CatBoostClassifier(
        iterations=5000,
        **params
    )


    model.fit(
        X_train,
        y_train,

        cat_features=categorical,

        verbose=False
    )


    models.append(model)


    probs = model.predict_proba(
        X_test
    )

    test_probs.append(probs)



# ==========================
# Test ensemble predictions
# ==========================

ensemble_probs = np.mean(
    test_probs,
    axis=0
)


pred_indices = np.argmax(
    ensemble_probs,
    axis=1
)


final_predictions = models[0].classes_[
    pred_indices
]



# ==========================
# Training accuracy check
# ==========================

train_probs = []

for model in models:

    probs = model.predict_proba(
        X_train
    )

    train_probs.append(probs)



ensemble_train_probs = np.mean(
    train_probs,
    axis=0
)


train_pred_indices = np.argmax(
    ensemble_train_probs,
    axis=1
)


train_predictions = models[0].classes_[
    train_pred_indices
]


train_accuracy = accuracy_score(
    y_train,
    train_predictions
)


print(
    f"In-sample training accuracy: {train_accuracy:.5f}"
)



# ==========================
# Submission
# ==========================

submission = sample_submission.copy()

submission["health_condition"] = final_predictions


submission.to_csv(
    "submission.csv",
    index=False
)


submission.head()

Training seed 42
Training seed 123
Training seed 456
Training seed 789
Training seed 999
In-sample training accuracy: 0.98131


,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Hyper parameter tuned CatBoost for accuracy

In [27]:
def objective(trial):

    params = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "depth": trial.suggest_int(
            "depth", 4, 10
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1.0, 10.0, log=True
        ),

        "border_count": trial.suggest_int(
            "border_count", 32, 255
        ),

        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 10.0
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 1.0
        ),

        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf", 1, 50
        ),

        "grow_policy": trial.suggest_categorical(
            "grow_policy",
            [
                "SymmetricTree",
                "Depthwise",
                "Lossguide"
            ]
        ),

        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            [
                "Balanced",
                None
            ]
        ),

        "loss_function": "MultiClass",
        "eval_metric": "Accuracy",

        "task_type": "GPU",
        "devices": "0",

        "random_seed": 42,
        "verbose": False
    }


    scores = []


    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train)
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]


        model = CatBoostClassifier(
            iterations=5000,
            **params
        )


        model.fit(
            X_tr,
            y_tr,

            cat_features=categorical,

            eval_set=(
                X_val,
                y_val
            ),

            early_stopping_rounds=100,

            use_best_model=True
        )


        preds = model.predict(X_val)


        score = accuracy_score(
            y_val,
            preds
        )


        scores.append(score)


        trial.report(
            np.mean(scores),
            fold
        )


        if trial.should_prune():
            raise optuna.TrialPruned()


    return np.mean(scores)



study = optuna.create_study(
    direction="maximize",

    pruner=optuna.pruners.MedianPruner(
        n_warmup_steps=2
    )
)


study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)



print(
    f"Best CV accuracy: {study.best_value:.5f}"
)


print("\nBest parameters:")
for k, v in study.best_params.items():
    print(k, ":", v)

[I 2026-07-24 18:54:11,500] A new study created in memory with name: no-name-7624fa67-f669-4035-8355-599f08cd1458


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-07-24 18:55:25,844] Trial 0 finished with value: 0.9373688567521012 and parameters: {'learning_rate': 0.10800656257275325, 'depth': 4, 'l2_leaf_reg': 9.062192724031084, 'border_count': 165, 'random_strength': 8.796931278776128, 'bagging_temperature': 0.721166692598559, 'min_data_in_leaf': 39, 'grow_policy': 'Depthwise', 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.9373688567521012.
[I 2026-07-24 18:57:02,519] Trial 1 finished with value: 0.966820172438571 and parameters: {'learning_rate': 0.02317490306574493, 'depth': 4, 'l2_leaf_reg': 4.185272505077933, 'border_count': 234, 'random_strength': 9.39320898879071, 'bagging_temperature': 0.8772927795408179, 'min_data_in_leaf': 12, 'grow_policy': 'Lossguide', 'auto_class_weights': None}. Best is trial 1 with value: 0.966820172438571.
[I 2026-07-24 18:58:38,343] Trial 2 finished with value: 0.9665318042652105 and parameters: {'learning_rate': 0.013397766193787623, 'depth': 7, 'l2_leaf_reg': 5.5388093573357144, 'bo

In [28]:
best_params = study.best_params


final_params = {
    **best_params,

    "loss_function": "MultiClass",
    "eval_metric": "Accuracy",

    # same as Optuna objective
    "auto_class_weights": None,

    "task_type": "GPU",
    "devices": "0",

    "verbose": False
}


seeds = [
    42,
    123,
    456,
    789,
    999
]


models = []
test_probs = []


for seed in seeds:

    print(f"Training seed {seed}")

    params = {
        **final_params,
        "random_seed": seed
    }


    model = CatBoostClassifier(
        iterations=2000,
        **params
    )


    model.fit(
        X_train,
        y_train,

        cat_features=categorical,

        verbose=False
    )


    models.append(model)


    probs = model.predict_proba(
        X_test
    )

    test_probs.append(probs)

Training seed 42
Training seed 123
Training seed 456
Training seed 789
Training seed 999


In [29]:
ensemble_probs = np.mean(
    test_probs,
    axis=0
)

# Get class indices
pred_indices = np.argmax(
    ensemble_probs,
    axis=1
)

# Convert back to original labels
final_predictions = models[0].classes_[
    pred_indices
]

train_probs = []

for model in models:
    probs = model.predict_proba(
        X_train
    )
    train_probs.append(probs)


# Average training probabilities
ensemble_train_probs = np.mean(
    train_probs,
    axis=0
)


# Convert to predictions
train_pred_indices = np.argmax(
    ensemble_train_probs,
    axis=1
)


# Convert indices back to labels
train_predictions = models[0].classes_[
    train_pred_indices
]


train_accuracy = accuracy_score(
    y_train,
    train_predictions
)


print(
    f"In-sample training accuracy: {train_accuracy:.5f}"
)


submission = sample_submission.copy()

submission["health_condition"] = final_predictions


submission.to_csv(
    "submission.csv",
    index=False
)


submission.head()

In-sample training accuracy: 0.97161


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [23]:
print(
    pd.Series(final_predictions).value_counts(normalize=True)
)

print(
    y_train.value_counts(normalize=True)
)

at-risk      0.873868
unhealthy    0.073771
fit          0.052361
Name: proportion, dtype: float64
health_condition
at-risk      0.858675
unhealthy    0.083647
fit          0.057678
Name: proportion, dtype: float64


# LightGBM

In [30]:
X_train[categorical] = X_train[categorical].astype("category")
X_test[categorical] = X_test[categorical].astype("category")

In [ ]:
def objective(trial):

    params = {

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            16,
            256
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            200
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10.0,
            log=True
        ),

        "max_bin": trial.suggest_int(
            "max_bin",
            64,
            255
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            0.0,
            5.0
        ),

        "boosting_type": trial.suggest_categorical(
            "boosting_type",
            [
                "gbdt",
                "dart"
            ]
        ),

        "objective": "multiclass",

        "device": "gpu",

        "verbosity": -1,

        "random_state": 42
    }


    scores = []


    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train)
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]


        model = LGBMClassifier(
            n_estimators=5000,
            **params
        )


        model.fit(
            X_tr,
            y_tr,

            eval_set=[
                (
                    X_val,
                    y_val
                )
            ],

            callbacks=[
                early_stopping(
                    stopping_rounds=100,
                    verbose=False
                )
            ]
        )


        preds = model.predict(
            X_val
        )


        score = accuracy_score(
            y_val,
            preds
        )


        scores.append(score)


        trial.report(
            np.mean(scores),
            fold
        )


        if trial.should_prune():
            raise optuna.TrialPruned()


    return np.mean(scores)



study = optuna.create_study(
    direction="maximize",

    pruner=optuna.pruners.MedianPruner(
        n_warmup_steps=2
    )
)


study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)



print(
    f"Best CV accuracy: {study.best_value:.5f}"
)


print("\nBest parameters:")

for k, v in study.best_params.items():
    print(
        k,
        ":",
        v
    )

[I 2026-07-24 21:12:43,410] A new study created in memory with name: no-name-7da1cada-d62b-4b7b-8801-cf7be4ed3182


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lightgbm/callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


In [ ]:
best_params = study.best_params


final_params = {
    **best_params,

    "objective": "multiclass",

    "device": "gpu",

    "verbosity": -1
}


seeds = [
    42,
    123,
    456,
    789,
    999
]


models = []
test_probs = []


for seed in seeds:

    print(
        f"Training seed {seed}"
    )


    params = {
        **final_params,
        "random_state": seed
    }


    model = LGBMClassifier(
        n_estimators=5000,
        **params
    )


    model.fit(
        X_train,
        y_train
    )


    models.append(model)


    probs = model.predict_proba(
        X_test
    )


    test_probs.append(
        probs
    )

In [ ]:
ensemble_probs = np.mean(
    test_probs,
    axis=0
)

# Get class indices
pred_indices = np.argmax(
    ensemble_probs,
    axis=1
)

# Convert back to original labels
final_predictions = models[0].classes_[
    pred_indices
]

train_probs = []

for model in models:
    probs = model.predict_proba(
        X_train
    )
    train_probs.append(probs)


# Average training probabilities
ensemble_train_probs = np.mean(
    train_probs,
    axis=0
)


# Convert to predictions
train_pred_indices = np.argmax(
    ensemble_train_probs,
    axis=1
)


# Convert indices back to labels
train_predictions = models[0].classes_[
    train_pred_indices
]


train_accuracy = accuracy_score(
    y_train,
    train_predictions
)


print(
    f"In-sample training accuracy: {train_accuracy:.5f}"
)


submission = sample_submission.copy()

submission["health_condition"] = final_predictions


submission.to_csv(
    "submission.csv",
    index=False
)


submission.head()